# Tutorial 6 — Feature Engineering

This notebook derives **patient-level features** from the three validated OMOP tables produced in
Tutorials 4 and 5. The result is a single ML-ready feature table: one row per patient, one column
per derived feature.

Feature engineering runs as a **Code Object** on the Rhino client — the same mechanism
introduced in Tutorial 3. The difference here is that this Code Object reads **three inputs
simultaneously** (OMOP Person, Visit Occurrence, and Procedure Occurrence), joins them on
`person_id` entirely on the client, and produces a flat feature table ready for model training.

**Can I do this in the UI instead?**
Yes — you can create and run this Code Object from the FCP Dashboard. Navigate to Dashboard →
Code → New Code Object → Python Code, paste the script below, assign all three input schemas,
and click Run.

---
**Prerequisites:**
- Tutorial 4 complete — OMOP dataset UIDs required in the Configuration cell
- Tutorial 5 complete — validation confirmed the OMOP outputs are correct
- Login information (username & password)

**Inputs:** Three validated OMOP datasets (Person, Visit Occurrence, Procedure Occurrence)

**Outputs:** One patient-level feature dataset registered on the FCP. UID carried forward for model training.

## Step 1: Configuration

In [ ]:
import json
import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeObjectRunInput,
    CodeTypes,
)
from rhino_health.lib.endpoints.code_run.code_run_dataclass import CodeRunStatus
import rhino_health.lib.metrics
from getpass import getpass
from rhino_health import ApiEnvironment

# From Tutorial 4 — paste your OMOP dataset and schema UIDs here
PROJECT_UID        = "<YOUR_PROJECT_UID>"   # REPLACE
OMOP_PERSON_UID    = "<OMOP_PERSON_UID>"    # REPLACE
OMOP_VISIT_UID     = "<OMOP_VISIT_UID>"     # REPLACE
OMOP_PROCEDURE_UID = "<OMOP_PROCEDURE_UID>" # REPLACE

# Verify all values have been filled in
required = {
    "PROJECT_UID":        PROJECT_UID,
    "OMOP_PERSON_UID":    OMOP_PERSON_UID,
    "OMOP_VISIT_UID":     OMOP_VISIT_UID,
    "OMOP_PROCEDURE_UID": OMOP_PROCEDURE_UID,
}
for name, val in required.items():
    if val.startswith("<"):
        raise ValueError(f"Please fill in {name} before running this cell.")
    print(f"{name} = {val}")

## Step 2: Initialize Shared Utilities

Run this cell once. It defines helper functions used throughout this notebook.

> Be sure to replace `my_username` with your Rhino FCP username!

In [ ]:
def authenticate():
    """Authenticate and return a session."""
    my_username = "<YOUR_USERNAME>"  # REPLACE
    if my_username in ("", "<YOUR_EMAIL>", "<YOUR_USERNAME>"):
        raise ValueError("Please fill in your username in the authenticate() function.")
    session = rh.login(
        username=my_username,
        password=getpass(),
        rhino_api_url=ApiEnvironment.PROD_AWS_URL,  # e.g., STAGING_AWS_URL, SOLUTIONS_GCP_URL
    )
    print(f"Logged in as <{my_username}>.")
    return session


def register_or_reuse_code_object(session, name, description, input_schema_uids, project_uid):
    """
    Register a Generalized Compute Code Object if one with this name doesn't already exist.

    Uses the generic-python-runner container. The actual Python script is passed at run time
    via run_params — not embedded here — so the same Code Object can be re-run with any script
    without re-registration.
    """
    existing = session.code_object.get_code_object_by_name(name, project_uid=project_uid)
    if existing:
        print(f"Code Object '{name}' already exists — reusing: {existing.uid}")
        return existing

    co = session.code_object.create_code_object(CodeObjectCreateInput(
        name=name,
        description=description,
        project_uid=project_uid,
        code_type=CodeTypes.GENERALIZED_COMPUTE,
        config={
            "container_image_uri": session.get_container_image_uri(
                "generic-python-runner", rhino_common_image=True
            ),
        },
        input_data_schema_uids=input_schema_uids,
        output_data_schema_uids=[None],  # auto-infer output schema from /output/dataset.csv
    ))
    print(f"Code Object registered: {co.uid}")
    return co


def run_and_poll(session, code_object_uid, input_dataset_uids, output_name,
                 run_params=None, max_wait=600):
    """
    Execute a Code Object and wait for completion.

    input_dataset_uids: list of dataset UIDs, one per input slot (order matches registration).
    run_params: dict passed to the container at runtime — use {"code": "..."} to supply the
                Python script for generic-python-runner.
    Inside the container: slot 0 → /input/0/dataset.csv, slot 1 → /input/1/dataset.csv, etc.
    """
    async_response = session.code_object.run_code_object(CodeObjectRunInput(
        code_object_uid=code_object_uid,
        input_dataset_uids=[input_dataset_uids],  # [[uid0, uid1, uid2]] — one iteration, all slots
        output_dataset_naming_templates=[output_name],
        run_params=json.dumps(run_params) if run_params else None,
        timeout_seconds=max_wait,
    ))

    code_run_uid = async_response.code_run_uid
    print(f"Run initiated: {code_run_uid}")
    print("FCP UI: Dashboard → Code Runs → find this UID to monitor progress")

    code_run = session.code_run.get_code_run(code_run_uid)
    result = code_run.wait_for_completion(timeout_seconds=max_wait, print_progress=True)

    if result.status not in (CodeRunStatus.COMPLETED, CodeRunStatus.HALTED_SUCCESS):
        raise RuntimeError(f"Run ended with status: {result.status.value}")

    print("Run completed successfully.")
    return result


print("Utility functions loaded.")

In [ ]:
# Log in to Rhino FCP and create a session object to interact with the API
session = authenticate()

---
## Step 3: Define the Feature Engineering Script

The script below runs **inside the `generic-python-runner` container on the Rhino client**.
It is passed to the container at **run time** via `run_params["code"]` — it is not embedded
in the Code Object definition. This means you can update the script and re-run without
deleting or re-registering the Code Object.

**Input paths (by slot order, mounted by the platform):**
- `/input/0/dataset.csv` — OMOP Person
- `/input/1/dataset.csv` — OMOP Visit Occurrence
- `/input/2/dataset.csv` — OMOP Procedure Occurrence

**Output:** `/output/dataset.csv` — one row per patient, one column per derived feature.

**Features derived:**
- `age` — calculated as `2024 − year_of_birth`
- `is_male` / `is_female` — binary flags from OMOP gender concept IDs (8507, 8532)
- `visit_count` — total visits per patient
- `has_inpatient_visit` / `has_outpatient_visit` / `has_emergency_visit` — visit type flags
- `days_of_clinical_history` — days between first and last visit
- `procedure_count` — total procedures per patient
- `unique_procedure_count` — distinct procedure concept IDs per patient

**Data quality handling:**
The script validates all three tables before joining. Rows with missing required fields
(`person_id`, `year_of_birth`) and visit/procedure rows referencing a `person_id` not present
in the Person table are dropped and logged. Patients with no matching visits or procedures
are kept and receive zeros for those features.

In [ ]:
FEATURE_ENGINEERING_CODE = '''
import pandas as pd
import os

errors = []

# The platform mounts each input dataset at an indexed path.
# Slot order matches the input_schema_uids list set when creating the Code Object:
#   /input/0/dataset.csv → OMOP Person
#   /input/1/dataset.csv → OMOP Visit Occurrence
#   /input/2/dataset.csv → OMOP Procedure Occurrence
person_df = pd.read_csv("/input/0/dataset.csv")
visit_df  = pd.read_csv("/input/1/dataset.csv")
proc_df   = pd.read_csv("/input/2/dataset.csv")

print("Loaded: person={} rows, visits={} rows, procedures={} rows".format(
    len(person_df), len(visit_df), len(proc_df)
))

# --- Validate Person table ---
n_before = len(person_df)
person_df = person_df[person_df["person_id"].notna()]
if len(person_df) < n_before:
    errors.append("Person: dropped {} row(s) with null person_id".format(n_before - len(person_df)))

n_before = len(person_df)
person_df = person_df[person_df["year_of_birth"].notna()]
if len(person_df) < n_before:
    errors.append("Person: dropped {} row(s) with null year_of_birth - age cannot be derived".format(
        n_before - len(person_df)
    ))

valid_person_ids = set(person_df["person_id"].unique())
print("Valid person IDs after Person table validation: {}".format(len(valid_person_ids)))

# --- Validate Visit Occurrence table ---
n_before = len(visit_df)
visit_df = visit_df[visit_df["person_id"].notna()]
if len(visit_df) < n_before:
    errors.append("Visits: dropped {} row(s) with null person_id".format(n_before - len(visit_df)))

orphaned_visits = ~visit_df["person_id"].isin(valid_person_ids)
if orphaned_visits.sum():
    errors.append("Visits: dropped {} row(s) referencing a person_id not found in the Person table".format(
        orphaned_visits.sum()
    ))
    visit_df = visit_df[~orphaned_visits]

# --- Validate Procedure Occurrence table ---
n_before = len(proc_df)
proc_df = proc_df[proc_df["person_id"].notna()]
if len(proc_df) < n_before:
    errors.append("Procedures: dropped {} row(s) with null person_id".format(n_before - len(proc_df)))

orphaned_procs = ~proc_df["person_id"].isin(valid_person_ids)
if orphaned_procs.sum():
    errors.append("Procedures: dropped {} row(s) referencing a person_id not found in the Person table".format(
        orphaned_procs.sum()
    ))
    proc_df = proc_df[~orphaned_procs]

# --- Patient-level base features (from OMOP Person) ---
features = person_df[["person_id", "year_of_birth", "gender_concept_id",
                       "race_concept_id", "ethnicity_concept_id"]].copy()

REFERENCE_YEAR = 2024
features["age"] = (REFERENCE_YEAR - features["year_of_birth"]).astype(int)

# Cast to int — OMOP IDs may be float64 after harmonization
features["person_id"] = features["person_id"].astype(int)
for col in ["gender_concept_id", "race_concept_id", "ethnicity_concept_id"]:
    features[col] = features[col].fillna(0).astype(int)

features["is_male"]   = (features["gender_concept_id"] == 8507).astype(int)
features["is_female"] = (features["gender_concept_id"] == 8532).astype(int)

# --- Visit-level features ---
visit_agg = visit_df.groupby("person_id").agg(
    visit_count=("visit_occurrence_id", "count"),
    has_inpatient_visit=("visit_concept_id", lambda x: int((x == 9201).any())),
    has_outpatient_visit=("visit_concept_id", lambda x: int((x == 9202).any())),
    has_emergency_visit=("visit_concept_id", lambda x: int((x == 9203).any())),
).reset_index()

visit_df["visit_start_date"] = pd.to_datetime(visit_df["visit_start_date"], errors="coerce")
date_span = visit_df.groupby("person_id")["visit_start_date"].agg(
    lambda x: (x.max() - x.min()).days
).reset_index()
date_span.columns = ["person_id", "days_of_clinical_history"]
visit_agg = visit_agg.merge(date_span, on="person_id", how="left")

# --- Procedure-level features ---
proc_agg = proc_df.groupby("person_id").agg(
    procedure_count=("procedure_occurrence_id", "count"),
    unique_procedure_count=("procedure_concept_id", "nunique"),
).reset_index()

# --- Join ---
features = features.merge(visit_agg, on="person_id", how="left")
features = features.merge(proc_agg, on="person_id", how="left")

int_fill_cols = [
    "visit_count", "has_inpatient_visit", "has_outpatient_visit",
    "has_emergency_visit", "procedure_count", "unique_procedure_count",
    "days_of_clinical_history",
]
for col in int_fill_cols:
    if col in features.columns:
        features[col] = features[col].fillna(0).astype(int)

features = features.drop(columns=["year_of_birth"], errors="ignore")

print("Shape: {} rows x {} columns".format(len(features), len(features.columns)))
print("Age: min={}, max={}, mean={:.1f}".format(
    features["age"].min(), features["age"].max(), features["age"].mean()
))
print("Avg visits per patient: {:.1f}".format(features["visit_count"].mean()))
print("Avg procedures per patient: {:.1f}".format(features["procedure_count"].mean()))

if errors:
    print("DATA QUALITY WARNINGS - {} issue(s) found:".format(len(errors)))
    for i, msg in enumerate(errors, 1):
        print("  [{}] {}".format(i, msg))
else:
    print("No data quality issues detected.")

# Write CSV so downstream code objects can mount it as /input/dataset.csv.
os.makedirs("/output", exist_ok=True)
features.to_csv("/output/dataset.csv", index=False)
print("Output written: /output/dataset.csv")

# generic-python-runner collects outputs via the outputs variable.
outputs = [[features]]
'''

---
## Step 4: Register and Run

This step registers the **Generalized Compute** Code Object on the FCP (once) and runs it.
The `generic-python-runner` container is specified at registration — the Python script is
passed separately as `run_params["code"]` each time the Code Object is run, so you can
update the script in Step 3 and re-run this step without re-registering.

> **If the Code Object already exists from a previous run**, it will be reused automatically.
> Re-running `cell-register` with a changed script has no effect on the stored definition —
> just re-run `cell-run` to pick up the new code.

> **If you see a 503 build-unavailable error**, the platform's container build service is
> temporarily down. Wait a few minutes and try again, or contact support@rhinohealth.com.

In [ ]:
# Look up the actual schema UID for each OMOP dataset.
# Passing real schema UIDs (not None) ensures the platform creates genuine input slots
# and mounts each dataset at its indexed path (/input/0/, /input/1/, /input/2/).
omop_person_schema_uid = str(session.dataset.get_dataset(OMOP_PERSON_UID).data_schema_uid)
omop_visit_schema_uid  = str(session.dataset.get_dataset(OMOP_VISIT_UID).data_schema_uid)
omop_proc_schema_uid   = str(session.dataset.get_dataset(OMOP_PROCEDURE_UID).data_schema_uid)

print(f"OMOP Person schema:    {omop_person_schema_uid}")
print(f"OMOP Visit schema:     {omop_visit_schema_uid}")
print(f"OMOP Procedure schema: {omop_proc_schema_uid}")

feature_co = register_or_reuse_code_object(
    session,
    name="Feature Engineering — Patient Features",
    description="Derive patient-level features from OMOP Person, Visit Occurrence, and Procedure Occurrence.",
    input_schema_uids=[omop_person_schema_uid, omop_visit_schema_uid, omop_proc_schema_uid],
    project_uid=PROJECT_UID,
)

In [ ]:
feature_run = run_and_poll(
    session,
    code_object_uid=feature_co.uid,
    # One UID per input slot — order must match input_schema_uids in cell-register
    input_dataset_uids=[
        OMOP_PERSON_UID,
        OMOP_VISIT_UID,
        OMOP_PROCEDURE_UID,
    ],
    output_name="Patient Features — Site A",
    # The script is passed at run time; the generic-python-runner container executes it.
    run_params={"code": FEATURE_ENGINEERING_CODE},
)

feature_ds = session.dataset.get_dataset(feature_run.output_dataset_uids.root[0].root[0].root[0])
FEATURE_DATASET_UID        = feature_ds.uid
FEATURE_DATASET_SCHEMA_UID = feature_ds.data_schema_uid
print(f"\nFeature dataset UID:  {FEATURE_DATASET_UID}")
print(f"Auto-inferred schema: {FEATURE_DATASET_SCHEMA_UID}")

---
## Step 5: Verify Output with Federated Analytics

Run federated metrics against the feature dataset to confirm the output looks correct.
All checks run as aggregate queries — no individual rows are returned.

> If `FEATURE_DATASET_UID` is not defined (e.g., you are re-running this section after a kernel
> restart), paste the UID from the summary printed in Step 4 before running this cell.

In [ ]:
from rhino_health.lib.metrics import Count, Mean, StandardDeviation, Sum

# --- Total patients ---
result = session.dataset.get_dataset_metric(FEATURE_DATASET_UID, Count(variable="person_id"))
n_patients = result.output["count"]
print(f"Total patients: {n_patients:,}")
print()

# --- Gender breakdown (OMOP concept IDs: 8532=Female, 8507=Male, 0=Unknown) ---
print("Gender breakdown:")
gender_result = session.dataset.get_dataset_metric(
    FEATURE_DATASET_UID,
    Count(variable="gender_concept_id", group_by={"groupings": ["gender_concept_id"]}),
)
gender_labels = {"8532": "Female", "8507": "Male", "0": "Unknown/Other"}
for concept_id, counts in sorted(gender_result.output.items()):
    label = gender_labels.get(str(int(float(concept_id))), f"Concept {concept_id}")
    print(f"  {label:<16} {counts['count']:>5,}")
print()

# --- Age distribution ---
print("Age distribution:")
age_mean = session.dataset.get_dataset_metric(FEATURE_DATASET_UID, Mean(variable="age"))
age_std  = session.dataset.get_dataset_metric(FEATURE_DATASET_UID, StandardDeviation(variable="age"))
print(f"  Mean:   {age_mean.output['mean']:.1f}")
print(f"  StdDev: {age_std.output['stddev']:.1f}")
print()

# --- Visit totals and type breakdown ---
total_visits = session.dataset.get_dataset_metric(FEATURE_DATASET_UID, Sum(variable="visit_count"))
print(f"Total visits: {int(total_visits.output['sum']):,}")
print()

print("Patients with each visit type:")
for col, label in [
    ("has_inpatient_visit",  "Inpatient"),
    ("has_outpatient_visit", "Outpatient"),
    ("has_emergency_visit",  "Emergency"),
]:
    r = session.dataset.get_dataset_metric(FEATURE_DATASET_UID, Sum(variable=col))
    n = int(r.output["sum"])
    print(f"  {label:<12} {n:>5,}  ({n / n_patients * 100:.1f}% of patients)")
print()

# --- Procedure totals ---
total_procs = session.dataset.get_dataset_metric(FEATURE_DATASET_UID, Sum(variable="procedure_count"))
print(f"Total procedures: {int(total_procs.output['sum']):,}")
print()

print("See Datasets > Analytics for more info")


---
## Step 6: FCP UI — What to Check After Running This Notebook

1. **Code Object** → Dashboard → Projects → [Your Project] → **Code**
   - `Feature Engineering — Patient Features` appears in the list
   - Three input schemas are listed (Person, Visit Occurrence, Procedure Occurrence)
   - This Code Object is reusable — run it at additional sites with a single API call

2. **Code Run** → Dashboard → Projects → [Your Project] → **Code Runs**
   - A completed run appears under `Feature Engineering — Patient Features`
   - Click to see: all three input datasets listed, one output dataset, timing, and container logs
   - The logs show row counts, feature summaries, and the data quality warning block at the end
   - If any warnings were printed (e.g., dropped rows due to missing person_id), review them here

3. **Output Dataset** → Dashboard → Projects → [Your Project] → **Datasets**
   - `Patient Features — Site A` appears
   - Row count should match the number of valid patients reported in the run logs
   - Click → **Analytics** tab → inspect distributions for `age`, `visit_count`, `procedure_count`
   - The auto-inferred schema will list all derived feature columns with their inferred types

---
## Summary — Copy These UIDs

In [ ]:
print("=" * 65)
print("  Tutorial 6 Complete — save these UIDs")
print("=" * 65)
print(f"FEATURE_DATASET_UID        = '{FEATURE_DATASET_UID}'")
print(f"FEATURE_DATASET_SCHEMA_UID = '{FEATURE_DATASET_SCHEMA_UID}'")
print("=" * 65)
print("\nContinue to: Tutorial 7 - Cohort Selection")